In [0]:
# CELL 1 - did everything land?
LANDING = "/Volumes/lumora/raw/landing"

def count_files(path):
    """Recursively walk a volume path and count files + bytes."""
    n, size = 0, 0
    for f in dbutils.fs.ls(path):
        if f.isDir():
            sub_n, sub_size = count_files(f.path)
            n += sub_n; size += sub_size
        else:
            n += 1; size += f.size
    return n, size

for folder in ["ads/meta", "ads/google", "ads/tiktok", "ads/offline_costs",
               "web/sessions", "ecommerce/orders", "ecommerce/order_events",
               "crm/customers_snapshot", "crm/email_sends", "reference"]:
    n, size = count_files(f"{LANDING}/{folder}")
    print(f"{folder:32s} {n:4d} files   {size/1024/1024:8.1f} MB")

In [0]:
# CELL 2 - Auto Loader: Meta ads -> bronze
from pyspark.sql import functions as F
import re

CHECKPOINTS = "/Volumes/lumora/raw/checkpoints"
SOURCE = "meta_ads"

def clean_names(df):
    """Delta forbids ' ,;{}()\\n\\t=' in column names. Rename mechanically:
    lowercase, non-alphanumeric -> '_', collapse repeats, strip edges.
    No business renaming here - that is silver's job."""
    for old in df.columns:
        new = re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z]+", "_", old)).strip("_").lower()
        if new != old:
            df = df.withColumnRenamed(old, new)
    return df

# readStream = a DataFrame that represents data arriving over time
df = (
    spark.readStream
        .format("cloudFiles")                                  # <- Auto Loader
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{CHECKPOINTS}/{SOURCE}/schema")
        .option("cloudFiles.inferColumnTypes", "false")        # everything as STRING
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .option("rescuedDataColumn", "_rescued_data")          # nothing is lost silently
        .load(f"{LANDING}/ads/meta/")
)

df = clean_names(df)

# Ingestion metadata: WHERE each row came from and WHEN we read it.
# _metadata is a hidden column Spark exposes for file-based sources.
df = (
    df.withColumn("_source_file",   F.col("_metadata.file_path"))
      .withColumn("_file_modified", F.col("_metadata.file_modification_time"))
      .withColumn("_ingested_at",   F.current_timestamp())
      .withColumn("_source_system", F.lit("meta_ads"))
)

# writeStream ... trigger(availableNow=True): process everything pending, then stop.
query = (
    df.writeStream
      .option("checkpointLocation", f"{CHECKPOINTS}/{SOURCE}/checkpoint")
      .option("mergeSchema", "true")
      .trigger(availableNow=True)
      .toTable("lumora.bronze.meta_ads_raw")   # UC table, Delta by default
)

query.awaitTermination()   # block until this run finishes
print("done:", query.lastProgress["numInputRows"] if query.lastProgress else 0, "rows")

In [0]:
# CELL 3 - check the result
display(spark.table("lumora.bronze.meta_ads_raw").limit(5))

spark.sql("""
  SELECT _source_system,
         count(*)                    AS rows,
         count(DISTINCT _source_file) AS files,
         min(day) AS first_day, max(day) AS last_day
  FROM lumora.bronze.meta_ads_raw
  GROUP BY _source_system
""").display()